In [14]:
# Imports Libraries
import random
import heapq
from copy import deepcopy

In [15]:
# Global Configuration
StateDimension = 4  # Change this to 3 or 4 to switch between 8-puzzle and 15-puzzle
GoalState = list(range(1, StateDimension**2)) + [0]

Opposite = {'u': 'd', 'd': 'u', 'l': 'r', 'r': 'l'}

In [16]:
# Action functions
def Actions(state):
    return ['u', 'd', 'l', 'r']

def Result(state, action):
    i = state.index(0)
    new_state = list(state)
    row, col = divmod(i, StateDimension)
    target = None
    if action == 'u' and row > 0:
        target = i - StateDimension
    elif action == 'd' and row < StateDimension - 1:
        target = i + StateDimension
    elif action == 'l' and col > 0:
        target = i - 1
    elif action == 'r' and col < StateDimension - 1:
        target = i + 1
    if target is not None:
        new_state[i], new_state[target] = new_state[target], new_state[i]
        return new_state
    return state


def GoalTest(state):
    return state == GoalState

In [17]:
# Heuristics
def H_OutOfPlace(state):
    return sum(1 for i in range(len(state)) if state[i] != 0 and state[i] != GoalState[i])


def H_Manhattan(state):
    total = 0
    for i, tile in enumerate(state):
        if tile == 0:
            continue
        goal_i = GoalState.index(tile)
        r1, c1 = divmod(i, StateDimension)
        r2, c2 = divmod(goal_i, StateDimension)
        total += abs(r1 - r2) + abs(c1 - c2)
    return total

In [18]:
# Search node class
class Node:
    def __init__(self, state, parent=None, action=None, cost=0, heuristic=0):
        self.state = state
        self.parent = parent
        self.action = action
        self.cost = cost
        self.heuristic = heuristic

    def __lt__(self, other):
        return (self.cost + self.heuristic) < (other.cost + other.heuristic)

    def path(self):
        node, p = self, []
        while node:
            p.append(node)
            node = node.parent
        return list(reversed(p))

In [19]:
# BFS search
def BFS(start):
    frontier = [Node(start)]
    explored = set()
    nodes_expanded = 0
    while frontier:
        node = frontier.pop(0)
        if tuple(node.state) in explored:
            continue
        explored.add(tuple(node.state))
        nodes_expanded += 1
        if GoalTest(node.state):
            return node.path(), nodes_expanded
        for a in Actions(node.state):
            child_state = Result(node.state, a)
            if tuple(child_state) not in explored:
                frontier.append(Node(child_state, node, a))
    return None, nodes_expanded

In [20]:
# A* search
def A_star(start, heuristic_func):
    frontier = [Node(start, heuristic=heuristic_func(start))]
    explored = {}
    nodes_expanded = 0
    while frontier:
        node = heapq.heappop(frontier)
        if GoalTest(node.state):
            return node.path(), nodes_expanded
        key = tuple(node.state)
        if key in explored and explored[key] <= node.cost:
            continue
        explored[key] = node.cost
        nodes_expanded += 1
        for a in Actions(node.state):
            child_state = Result(node.state, a)
            if child_state == node.state:
                continue
            child = Node(child_state, node, a, node.cost + 1, heuristic_func(child_state))
            heapq.heappush(frontier, child)
    return None, nodes_expanded

In [21]:
# Generate random puzzle
def random_walk(steps):
    state = GoalState[:]
    last_action = None
    for _ in range(steps):
        options = [a for a in Actions(state) if a != Opposite.get(last_action)]
        action = random.choice(options)
        state = Result(state, action)
        last_action = action
    return state

In [32]:
# Generate and test puzzles
def run_experiments(dim):
    global StateDimension, GoalState
    StateDimension = dim
    GoalState = list(range(1, StateDimension ** 2)) + [0]

    step_counts = [5, 10, 20, 40, 80]
    problems = []
    for steps in step_counts:
        for _ in range(3):
            problems.append((random_walk(steps), steps))

    results = []
    for idx, (problem, steps) in enumerate(problems):
        entry = {'start_state': problem, 'steps_from_goal': steps}

        # BFS (skip for 4x4 if >10 steps)
        if StateDimension == 3 or step_counts[idx // 3] <= 10:
            path, expanded = BFS(problem)
            entry['BFS'] = {
                'solution': [node.action for node in path][1:] if path else None,
                'length': len(path) - 1 if path else None,
                'expanded': expanded
            }

        # A* Out of Place
        path, expanded = A_star(problem, H_OutOfPlace)
        entry['A*_OutOfPlace'] = {
            'solution': [node.action for node in path][1:] if path else None,
            'length': len(path) - 1 if path else None,
            'expanded': expanded
        }

        # A* Manhattan
        path, expanded = A_star(problem, H_Manhattan)
        entry['A*_Manhattan'] = {
            'solution': [node.action for node in path][1:] if path else None,
            'length': len(path) - 1 if path else None,
            'expanded': expanded
        }

        results.append(entry)
    return results

In [33]:
# Print output function
def print_results(results, dimension):
    print(f"\n=== Results for {dimension}x{dimension} Puzzle ===\n")
    for i, result in enumerate(results):
        print(f"--- Problem {i + 1} (Start State: {result['start_state']}, Steps from goal: {result['steps_from_goal']}) ---")


        if 'BFS' in result:
            bfs = result['BFS']
            print("BFS:")
            print(f"  Solution: {bfs['solution']}")
            print(f"  Length: {bfs['length']}")
            print(f"  Nodes Expanded: {bfs['expanded']}")

        a1 = result['A*_OutOfPlace']
        print("A* (Out of Place):")
        print(f"  Solution: {a1['solution']}")
        print(f"  Length: {a1['length']}")
        print(f"  Nodes Expanded: {a1['expanded']}")

        a2 = result['A*_Manhattan']
        print("A* (Manhattan):")
        print(f"  Solution: {a2['solution']}")
        print(f"  Length: {a2['length']}")
        print(f"  Nodes Expanded: {a2['expanded']}")

        print("\n" + "="*60 + "\n")

In [36]:
# Run and print 3x3 results
results_3x3 = run_experiments(3)
print_results(results_3x3, 3)


=== Results for 3x3 Puzzle ===

--- Problem 1 (Start State: [1, 2, 0, 4, 5, 3, 7, 8, 6], Steps from goal: 5) ---
BFS:
  Solution: ['d', 'd']
  Length: 2
  Nodes Expanded: 4
A* (Out of Place):
  Solution: ['d', 'd']
  Length: 2
  Nodes Expanded: 2
A* (Manhattan):
  Solution: ['d', 'd']
  Length: 2
  Nodes Expanded: 2


--- Problem 2 (Start State: [1, 0, 2, 4, 5, 3, 7, 8, 6], Steps from goal: 5) ---
BFS:
  Solution: ['r', 'd', 'd']
  Length: 3
  Nodes Expanded: 18
A* (Out of Place):
  Solution: ['r', 'd', 'd']
  Length: 3
  Nodes Expanded: 3
A* (Manhattan):
  Solution: ['r', 'd', 'd']
  Length: 3
  Nodes Expanded: 3


--- Problem 3 (Start State: [1, 2, 3, 4, 5, 6, 0, 7, 8], Steps from goal: 5) ---
BFS:
  Solution: ['r', 'r']
  Length: 2
  Nodes Expanded: 7
A* (Out of Place):
  Solution: ['r', 'r']
  Length: 2
  Nodes Expanded: 2
A* (Manhattan):
  Solution: ['r', 'r']
  Length: 2
  Nodes Expanded: 2


--- Problem 4 (Start State: [1, 2, 0, 4, 5, 3, 7, 8, 6], Steps from goal: 10) ---
BFS:


In [37]:
# Run and print 4x4 results
results_4x4 = run_experiments(4)
print_results(results_4x4, 4)


=== Results for 4x4 Puzzle ===

--- Problem 1 (Start State: [1, 2, 3, 4, 5, 6, 7, 8, 9, 10, 11, 0, 13, 14, 15, 12], Steps from goal: 5) ---
BFS:
  Solution: ['d']
  Length: 1
  Nodes Expanded: 3
A* (Out of Place):
  Solution: ['d']
  Length: 1
  Nodes Expanded: 1
A* (Manhattan):
  Solution: ['d']
  Length: 1
  Nodes Expanded: 1


--- Problem 2 (Start State: [1, 2, 3, 4, 5, 6, 7, 8, 9, 10, 12, 0, 13, 14, 11, 15], Steps from goal: 5) ---
BFS:
  Solution: ['l', 'd', 'r']
  Length: 3
  Nodes Expanded: 21
A* (Out of Place):
  Solution: ['l', 'd', 'r']
  Length: 3
  Nodes Expanded: 3
A* (Manhattan):
  Solution: ['l', 'd', 'r']
  Length: 3
  Nodes Expanded: 3


--- Problem 3 (Start State: [1, 2, 3, 4, 5, 6, 7, 8, 9, 10, 11, 12, 13, 14, 15, 0], Steps from goal: 5) ---
BFS:
  Solution: []
  Length: 0
  Nodes Expanded: 1
A* (Out of Place):
  Solution: []
  Length: 0
  Nodes Expanded: 0
A* (Manhattan):
  Solution: []
  Length: 0
  Nodes Expanded: 0


--- Problem 4 (Start State: [1, 2, 3, 4, 5, 6

Search is a fundamental tool in Artificial Intelligence for solving problems, planning, and decision-making. The experimental results with the 3x3 and 4x4 sliding puzzles demonstrate that as problem complexity increases, uninformed search methods like Breadth-First Search quickly become impractical due to exponential growth in the state space. As all 15 3x3 problems took around 1 second while all 15 4x4 problems took around 1 minute. In contrast, informed search methods like A* remain feasible for larger problems when paired with effective heuristics. The Manhattan Distance heuristic consistently outperforms the simpler Out-of-Place heuristic by reducing node expansions and finding shorter paths. These findings highlight the importance of incorporating domain knowledge into AI systems to guide search efficiently and scale to more complex real-world problems.